# 1. Setup

### 1.1 Install deps & packages

In [ ]:
%pip install numpy pandas matplotlib kagglehub
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import kagglehub
import shutil
import os

### 1.2 Download dataset

In [8]:
try:
    os.mkdir('original_dataset')
    kagglehub.dataset_download("nafiulislam490/bank-transaction-fraud-detection-dataset", output_dir='original_dataset')
except FileExistsError:
    pass

# Remove uesless metadata
shutil.rmtree('original_dataset/.complete/', ignore_errors=True)

print('Dataset downloaded')

# 2. Data pipeline

### 2.1 Load csv into dataframe

In [13]:
fraud_data = pd.read_csv('original_dataset/bank_fraud.csv')

print(f"Shape\n{fraud_data.shape}\n")
print(f"Description\n{fraud_data.describe()}\n")
print(f"Head\n{fraud_data.head()}")

In [14]:
fraud_data_original = fraud_data.copy(deep=True)

print("Records:", fraud_data_original.shape[0])
print("Columns:", fraud_data_original.shape[1])

print("\nColumn names and data types:")
print(fraud_data_original.dtypes)

print("\nData-type counts:")
print(fraud_data_original.dtypes.value_counts())

In [15]:
summary = pd.DataFrame({
    'dtype': fraud_data_original.dtypes,
    'sample_values': [fraud_data_original[col].dropna().unique()[:3] for col in fraud_data_original.columns]
})
print(summary.to_string())

                            dtype                                         sample_values
transaction_id             object         [TXN0000000001, TXN0000000002, TXN0000000003]
customer_id                object            [CUST00121959, CUST00146868, CUST00131933]
transaction_date           object                  [2023-08-17, 2024-02-06, 2024-06-28]
transaction_time           object                        [21:13:00, 05:16:00, 12:15:00]
hour_of_day                 int64                                           [21, 5, 12]
is_weekend                  int64                                                [0, 1]
is_night_transaction        int64                                                [0, 1]
country                    object                                     [USA, UK, Canada]
city                       object                             [London, New York, Delhi]
merchant_category          object                      [Grocery, Healthcare, Utilities]
payment_method             objec

# 3. Data Cleansing and Transformation

### 3.1 Dropping columns

Looking at the dataset, we can see the transaction ID is unique to every row, therefore it won't be useful in deriving any value.
We can use the customer contry to identify if the transaction took place in a high risk country, for this reason we can drop the city as it's too spesific.

In [ ]:
fraud_data = fraud_data.drop(columns=['transaction_id', 'country', 'city'])

fraud_data.head()

### 3.2 Normalising data

Since a lot of these columns are categories we can convert them to a number using a dict

In [ ]:
# A list of categorical columns
categorical_columns = ['merchant_category', 'payment_method', 'device_type', 'fraud_type']

# Create a dict of the enum values for each category
category_values = {}

for category in categorical_columns:
    category_values[category] = fraud_data[category].unique().tolist()

for i in range(fraud_data.shape[0]):
    for key in category_values:
        fraud_data.at[i, key] = category_values[key].index(fraud_data.loc[i][key])
    
fraud_data.head()

### 3.3 Creating a new feature for identifying high risk transactions

We can create a new feature called `high_risk` for what is roughly a high risk transaction, this would be defined by
- `account_age_years` <= 1
- `time_since_last_txn_hrs` <= 1
- `is_international` == 1
- `pin_changed_recently` == 1
- `transaction_amount` >= 100

In [ ]:
fraud_data['high_risk'] = np.where(
    (fraud_data['account_age_years'] <= 1) &
    (fraud_data['time_since_last_txn_hrs'] <= 1) &
    (fraud_data['is_international'] == 1) &
    (fraud_data['transaction_amount'] >= 100) &
    (fraud_data['pin_changed_recently'] == 1),
    1, 0)

filtered_df = fraud_data[fraud_data['high_risk'] == 1]

print(f"Found {filtered_df.shape[0]} high risk transactions")

fraud_data['high_risk'].describe()